<a href="https://colab.research.google.com/github/zixian0821-zoe/stochastic_search_and_optimization/blob/main/hw11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
from numpy.random import default_rng

def build_B(p):
    return np.triu(np.ones((p, p)))

def loss_func(theta, B):
    Btheta = B @ theta
    return np.dot(Btheta, Btheta) + 0.1 * np.sum(Btheta**3) + 0.01 * np.sum(Btheta**4)

def noisy_loss(theta, B, sigma, rng):
    return loss_func(theta, B) + sigma * rng.standard_normal()

def run_spsa(p, B, sigma, N_budget, rng):
    alpha = 0.602
    gamma = 0.101
    A = 100

    if sigma == 0:
        c = 0.1
    elif sigma == 1:
        c = 1.0
    else:
        c = 5.0

    theta_init = np.ones(p)
    n_pilot = 5
    g_mags = []

    for _ in range(n_pilot):
        delta = rng.choice([-1, 1], size=p).astype(float)
        c0 = c
        yp = noisy_loss(theta_init + c0 * delta, B, sigma, rng)
        ym = noisy_loss(theta_init - c0 * delta, B, sigma, rng)
        g = (yp - ym) / (2 * c0) * (1.0 / delta)
        g_mags.append(np.linalg.norm(g, ord=np.inf))

    g_avg = np.mean(g_mags)
    desired_change = 0.1
    a0 = desired_change / max(g_avg, 1e-10)
    a = a0 * (1 + A) ** alpha

    n_pilot_evals = n_pilot * 2
    n_iter_actual = (N_budget - n_pilot_evals) // 2

    theta = theta_init.copy()
    for k in range(n_iter_actual):
        a_k = a / (k + 1 + A) ** alpha
        c_k = c / (k + 1) ** gamma
        delta_k = rng.choice([-1, 1], size=p).astype(float)
        y_plus = noisy_loss(theta + c_k * delta_k, B, sigma, rng)
        y_minus = noisy_loss(theta - c_k * delta_k, B, sigma, rng)
        g_hat = (y_plus - y_minus) / (2 * c_k) * (1.0 / delta_k)
        theta = theta - a_k * g_hat

    return theta, loss_func(theta, B)

def run_san(p, B, sigma, N_budget, rng):
    iters_per_temp = 50

    if sigma == 0:
        total_steps = N_budget - 1
        T0 = 5.0
        lam = 0.85
        pert_std = 0.5
    elif sigma == 1:
        total_steps = (N_budget - 1) // 2
        T0 = 10.0
        lam = 0.88
        pert_std = 0.5
    else:
        total_steps = (N_budget - 1) // 2
        T0 = 50.0
        lam = 0.90
        pert_std = 0.5

    n_stages = total_steps // iters_per_temp
    c_b = 1.0

    theta_curr = np.ones(p)
    L_curr_stored = noisy_loss(theta_curr, B, sigma, rng)

    theta_best = theta_curr.copy()
    L_best_true = loss_func(theta_curr, B)

    T = T0
    for _ in range(n_stages):
        for _ in range(iters_per_temp):
            theta_new = theta_curr + pert_std * rng.standard_normal(p)
            L_new = noisy_loss(theta_new, B, sigma, rng)

            if sigma > 0:
                L_curr_meas = noisy_loss(theta_curr, B, sigma, rng)
            else:
                L_curr_meas = L_curr_stored

            delta = L_new - L_curr_meas

            accept = False
            if delta < 0:
                accept = True
            else:
                if T > 1e-15:
                    prob = np.exp(min(-delta / (c_b * T), 0))
                    if rng.random() < prob:
                        accept = True

            if accept:
                theta_curr = theta_new.copy()
                L_curr_stored = L_new

            L_true = loss_func(theta_curr, B)
            if L_true < L_best_true:
                L_best_true = L_true
                theta_best = theta_curr.copy()

        T = lam * T

    return theta_best, L_best_true

def run_experiment():
    p = 20
    B = build_B(p)
    N_budget = 2000
    n_reps = 40

    theta0 = np.ones(p)
    L0 = loss_func(theta0, B)
    print(f"Skewed-Quartic Loss Function (p = {p})")
    print(f"θ* = 0, L(θ*) = 0")
    print(f"θ₀ = [1, ..., 1]^T, L(θ₀) = {L0:.4f}")
    print(f"Total measurement budget N = {N_budget}")
    print(f"Replications = {n_reps}")
    print("=" * 70)

    sigma_values = [0, 1, 10]
    results = {}

    for sigma in sigma_values:
        print(f"\n--- σ = {sigma} ---")
        spsa_losses = []
        san_losses = []

        for rep in range(n_reps):
            seed_base = 42 + 1000 * sigma_values.index(sigma)

            rng_spsa = default_rng(seed_base + rep)
            _, L_spsa = run_spsa(p, B, sigma, N_budget, rng_spsa)
            spsa_losses.append(L_spsa)

            rng_san = default_rng(seed_base + rep + 500)
            _, L_san = run_san(p, B, sigma, N_budget, rng_san)
            san_losses.append(L_san)

            if rep % 10 == 0:
                print(f"  Rep {rep}: SPSA L={L_spsa:.4f}, SAN L={L_san:.4f}")

        spsa_losses = np.array(spsa_losses)
        san_losses = np.array(san_losses)

        results[sigma] = {
            "spsa_mean": np.mean(spsa_losses),
            "spsa_std": np.std(spsa_losses, ddof=1),
            "spsa_min": np.min(spsa_losses),
            "spsa_max": np.max(spsa_losses),
            "san_mean": np.mean(san_losses),
            "san_std": np.std(san_losses, ddof=1),
            "san_min": np.min(san_losses),
            "san_max": np.max(san_losses),
        }

    print(f"\n\n{'=' * 70}")
    print("SUMMARY: Sample Mean (Std) of L(θ̂_final) over 40 Replications")
    print(f"{'=' * 70}")
    print(f"{'σ':>4} | {'SPSA':>24} | {'SAN':>24}")
    print(f"{'':>4} | {'Mean (Std)':>24} | {'Mean (Std)':>24}")
    print("-" * 60)
    for sigma in sigma_values:
        r = results[sigma]
        print(f"{sigma:>4} | {r['spsa_mean']:>10.2f} ({r['spsa_std']:>8.2f}) | {r['san_mean']:>10.2f} ({r['san_std']:>8.2f})")

    print(f"\nL(θ₀) = {L0:.2f}")
    print("L(θ*) = 0.00")

    return results

results = run_experiment()

Skewed-Quartic Loss Function (p = 20)
θ* = 0, L(θ*) = 0
θ₀ = [1, ..., 1]^T, L(θ₀) = 14506.6600
Total measurement budget N = 2000
Replications = 40

--- σ = 0 ---
  Rep 0: SPSA L=32.4668, SAN L=8.0662
  Rep 10: SPSA L=33.2583, SAN L=7.9800
  Rep 20: SPSA L=43.4219, SAN L=11.3782
  Rep 30: SPSA L=45.8715, SAN L=9.4377

--- σ = 1 ---
  Rep 0: SPSA L=35.2699, SAN L=8.5221
  Rep 10: SPSA L=47.7683, SAN L=26.7589
  Rep 20: SPSA L=23.8521, SAN L=20.4301
  Rep 30: SPSA L=52.4253, SAN L=29.2270

--- σ = 10 ---
  Rep 0: SPSA L=25.2164, SAN L=30.7811
  Rep 10: SPSA L=81.6810, SAN L=17.6493
  Rep 20: SPSA L=76.9299, SAN L=81.0632
  Rep 30: SPSA L=18.0840, SAN L=50.5912


SUMMARY: Sample Mean (Std) of L(θ̂_final) over 40 Replications
   σ |                     SPSA |                      SAN
     |               Mean (Std) |               Mean (Std)
------------------------------------------------------------
   0 |      33.80 (   19.74) |       8.25 (    2.53)
   1 |      24.94 (   15.33) |      2

In [2]:
import math

a = b = A = B = 1
alpha = 0.8
k_values = [1000, 1_000_000]

def gains_A(k):
    ak = a / math.log(k + 1)
    bk = ak
    return ak, bk

def gains_B(k):
    ak = a / (k + 1 + A)
    bk = b / (math.sqrt(k + 1) * math.log(math.log(k + 1)))
    return ak, bk

def gains_C(k):
    ak = a / (k + 1 + A)**alpha
    bk = b / ((k + 1)**(alpha / 2) * math.log(k + 1))
    return ak, bk

def gains_D(k):
    ak = a / (k + 1 + A)**alpha
    bk = b / math.sqrt((k + 1)**alpha * math.log((k + 1)**(1 - alpha) + B))
    return ak, bk

funcs = {'A': gains_A, 'B': gains_B, 'C': gains_C, 'D': gains_D}
results = {}

for k in k_values:
    print(f"\nk = {k:,}")
    print(f"{'Cond':>5} | {'a_k':>14} | {'b_k':>14} | {'a_k * b_k':>14}")
    print("-" * 55)
    for label, func in funcs.items():
        ak, bk = func(k)
        results[(label, k)] = (ak, bk)
        print(f"{label:>5} | {ak:>14.6e} | {bk:>14.6e} | {ak * bk:>14.6e}")

print(f"\nDecay ratio (k=1,000,000 / k=1,000):")
print(f"{'Cond':>5} | {'a_k ratio':>14} | {'b_k ratio':>14}")
print("-" * 40)
for label in ['A', 'B', 'C', 'D']:
    ak1, bk1 = results[(label, 1000)]
    ak2, bk2 = results[(label, 1_000_000)]
    print(f"{label:>5} | {ak2/ak1:>14.6e} | {bk2/bk1:>14.6e}")


k = 1,000
 Cond |            a_k |            b_k |      a_k * b_k
-------------------------------------------------------
    A |   1.447439e-01 |   1.447439e-01 |   2.095079e-02
    B |   9.980040e-04 |   1.635304e-02 |   1.632040e-05
    C |   3.974713e-03 |   9.129071e-03 |   3.628544e-05
    D |   3.974713e-03 |   4.977141e-02 |   1.978271e-04

k = 1,000,000
 Cond |            a_k |            b_k |      a_k * b_k
-------------------------------------------------------
    A |   7.238241e-02 |   7.238241e-02 |   5.239213e-03
    B |   9.999980e-07 |   3.808373e-04 |   3.808365e-10
    C |   1.584891e-05 |   2.881594e-04 |   4.567012e-09
    D |   1.584891e-05 |   2.368893e-03 |   3.754436e-08

Decay ratio (k=1,000,000 / k=1,000):
 Cond |      a_k ratio |      b_k ratio
----------------------------------------
    A |   5.000723e-01 |   5.000723e-01
    B |   1.001998e-03 |   2.328847e-02
    C |   3.987434e-03 |   3.156503e-02
    D |   3.987434e-03 |   4.759544e-02


In [3]:
import math
from itertools import combinations

L_bar = [71, 86, 72, 63, 70]
n = [10, 10, 10, 6, 6]
s = 9.8
Q_alpha = 4.06
K = 5
nu = sum(n) - K
cf = (Q_alpha / math.sqrt(2)) * s

print(f"K={K}, nu={nu}, s={s}, Q={Q_alpha}, factor={cf:.4f}\n")

print(f"{'(i,j)':>7} | {'delta':>7} | {'|delta|':>7} | {'w_ij':>8} | result")
print("-" * 50)

for i, j in combinations(range(K), 2):
    d = L_bar[i] - L_bar[j]
    w = cf * math.sqrt(1/n[i] + 1/n[j])
    r = "REJECT" if abs(d) > w else "accept"
    print(f"  ({i+1},{j+1}) | {d:>7.1f} | {abs(d):>7.1f} | {w:>8.3f} | {r}")

print(f"\nPart (b): theta_2 vs all others")
for j in range(K):
    if j == 1:
        continue
    d = L_bar[1] - L_bar[j]
    w = cf * math.sqrt(1/n[1] + 1/n[j])
    print(f"  delta(2,{j+1}) = {d:+.1f}, |d|={abs(d):.1f} {'>' if abs(d)>w else '<='} w={w:.3f}")
print("theta_2 is ruled out as theta* candidate")

K=5, nu=37, s=9.8, Q=4.06, factor=28.1344

  (i,j) |   delta | |delta| |     w_ij | result
--------------------------------------------------
  (1,2) |   -15.0 |    15.0 |   12.582 | REJECT
  (1,3) |    -1.0 |     1.0 |   12.582 | accept
  (1,4) |     8.0 |     8.0 |   14.529 | accept
  (1,5) |     1.0 |     1.0 |   14.529 | accept
  (2,3) |    14.0 |    14.0 |   12.582 | REJECT
  (2,4) |    23.0 |    23.0 |   14.529 | REJECT
  (2,5) |    16.0 |    16.0 |   14.529 | REJECT
  (3,4) |     9.0 |     9.0 |   14.529 | accept
  (3,5) |     2.0 |     2.0 |   14.529 | accept
  (4,5) |    -7.0 |     7.0 |   16.243 | accept

Part (b): theta_2 vs all others
  delta(2,1) = +15.0, |d|=15.0 > w=12.582
  delta(2,3) = +14.0, |d|=14.0 > w=12.582
  delta(2,4) = +23.0, |d|=23.0 > w=14.529
  delta(2,5) = +16.0, |d|=16.0 > w=14.529
theta_2 is ruled out as theta* candidate
